In [1]:
import os
from groq import Groq
from dotenv import load_dotenv

In [14]:
load_dotenv()
def get_completion(prompt: str, model: str = "llama3-70b-8192"):
    groq = Groq(api_key=os.environ["GROQ_API_KEY"])
    messages = [{"role": "user", "content": prompt}]
    response = groq.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, 
    )
    return response.choices[0].message.content

In [7]:
from IPython.display import display, Markdown, Latex

In [11]:
PROMPT = """
Use the `diagrams` Python library for programmatically generating a GCS architectural diagram for a data ingestion 
pipeline that extracts XML data from MongoDB into GCS, parses the XML files to JSON, chunks the parsed files into .csv 
files, and indexes and embeds the chunks for dense vector search in ElasticSearch.

Return only a well-structured and well-documented Python code snippet between 3 backticks ```python and ```
"""

In [15]:
display(Markdown(get_completion(model="llama3-70b-8192", prompt=PROMPT)))

```python
from diagrams import Cluster, Diagram, Edge
from diagrams.onprem.database import MongoDB
from diagrams.onprem.storage import GCS
from diagrams.onprem.workflow import ApacheBeam
from diagrams.onprem.search import ElasticSearch
from diagrams.programming.framework import Python

with Diagram("Data Ingestion Pipeline", direction="TB"):
    """
    Data Ingestion Pipeline Architectural Diagram
    """

    # MongoDB Cluster
    with Cluster("MongoDB"):
        mongo = MongoDB("MongoDB")

    # GCS Bucket
    gcs = GCS("GCS Bucket")

    # Apache Beam Workflow
    with Cluster("Apache Beam"):
        beam = ApacheBeam("Data Ingestion")

    # Python Script for XML to JSON Parsing
    with Cluster("XML to JSON"):
        xml_to_json = Python("xml_to_json.py")

    # Chunking and Indexing
    with Cluster("Chunking and Indexing"):
        chunking = Python("chunking.py")
        indexing = Python("indexing.py")

    # ElasticSearch Cluster
    with Cluster("ElasticSearch"):
        es = ElasticSearch("ElasticSearch")

    # Edges
    mongo >> Edge(label="Extract XML Data") >> beam
    beam >> Edge(label="Store XML Files") >> gcs
    gcs >> Edge(label="Parse XML to JSON") >> xml_to_json
    xml_to_json >> Edge(label="Chunk JSON Files") >> chunking
    chunking >> Edge(label="Index and Embed Chunks") >> indexing
    indexing >> Edge(label="Index for Dense Vector Search") >> es
```

This code generates a diagram that represents the data ingestion pipeline architecture, including the components and their interactions. The diagram is structured into clusters for MongoDB, GCS, Apache Beam, XML to JSON parsing, chunking and indexing, and ElasticSearch. The edges represent the data flow between these components.

In [3]:
from diagrams import Diagram, Cluster
from diagrams.gcp.analytics import PubSub
from diagrams.gcp.compute import Functions
from diagrams.gcp.database import Datastore
from diagrams.gcp.storage import Storage
from diagrams.gcp.analytics import Dataflow
from diagrams.onprem.database import MongoDB
from diagrams.elastic.elasticsearch import Elasticsearch
from diagrams.gcp.operations import Monitoring

with Diagram("GCP Data Ingestion Pipeline", show=False) as diagram:
    pubsub = PubSub("Cloud Pub/Sub")
    monitoring = Monitoring("Cloud Monitoring")

    with Cluster("Data Sources"):
        mongo = MongoDB("MongoDB")

    with Cluster("Extraction and Storage"):
        extract_func = Functions("Extract XML")
        storage_xml = Storage("XML Storage")
        mongo >> extract_func >> storage_xml

    with Cluster("XML to JSON Conversion"):
        convert_func = Functions("XML to JSON")
        storage_json = Storage("JSON Storage")
        storage_xml >> convert_func >> storage_json

    with Cluster("Data Processing"):
        dataflow = Dataflow("Cloud Dataflow")
        storage_csv = Storage("CSV Storage")
        storage_json >> dataflow >> storage_csv

    with Cluster("Vector Database"):
        elastic = Elasticsearch("Elasticsearch")
        dataflow >> elastic

    pubsub >> [extract_func, convert_func, dataflow]
    monitoring >> [extract_func, convert_func, dataflow, elastic]

In [30]:
import operator
import functools
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    ToolMessage,
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import END, StateGraph, START
from langchain_core.tools import tool
from langchain_experimental.utilities import PythonREPL
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage, ToolMessage

In [25]:
def create_agent(llm, tools, system_message: str):
    """Create an agent."""
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful AI assistant, collaborating with other assistants."
                " Use the provided tools to progress towards answering the question."
                " If you are unable to fully answer, that's OK, another assistant with different tools "
                " will help where you left off. Execute what you can to make progress."
                " If you or any of the other assistants have the final answer or deliverable,"
                " prefix your response with FINAL ANSWER so the team knows to stop."
                " You have access to the following tools: {tool_names}.\n{system_message}",
            ),
            MessagesPlaceholder(variable_name="messages"),
        ]
    )
    prompt = prompt.partial(system_message=system_message)
    prompt = prompt.partial(tool_names=", ".join([tool.name for tool in tools]))
    return prompt | llm.bind_tools(tools)

In [26]:
repl = PythonREPL()

@tool
def python_repl(
    code: Annotated[str, "The python code to execute to generate your chart."],
):
    """Use this to execute python code. If you want to see the output of a value,
    you should print it out with `print(...)`. This is visible to the user."""
    try:
        result = repl.run(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    result_str = f"Successfully executed:\n```python\n{code}\n```\nStdout: {result}"
    return (
        result_str + "\n\nIf you have completed all tasks, respond with FINAL ANSWER."
    )

In [ ]:
# This defines the object that is passed between each node
# in the graph. We will create different nodes for each agent and tool
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    sender: str

# Helper function to create a node for a given agent
def agent_node(state, agent, name):
    result = agent.invoke(state)
    # We convert the agent output into a format that is suitable to append to the global state
    if isinstance(result, ToolMessage):
        pass
    else:
        result = AIMessage(**result.dict(exclude={"type", "name"}), name=name)
    return {
        "messages": [result],
        # Since we have a strict workflow, we can
        # track the sender so we know who to pass to next.
        "sender": name,
    }